# 02 · Transformation & Join

Loads all raw files, normalises to a common schema, resamples to month-end, joins onto a single master table, documents gaps, interpolates short gaps, and adds derived metrics.

**Input:**  `data/raw/*.csv`  
**Output:** `data/processed/master.csv`  
**Audit:**  `data/processed/gap_log_pre_imputation.csv`

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')

def load_boe(filename):
    """Tab-separated, metadata rows at top — find first row that starts with a date."""
    raw = (RAW_DIR / filename).read_text()
    lines = raw.splitlines()
    data_start = next(i for i, l in enumerate(lines) if l[:2].isdigit())
    df = pd.read_csv(
        pd.io.common.StringIO('\n'.join(lines[data_start:])),
        sep='\t', header=None, names=['date', 'value']
    )
    df['date']  = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)


def load_ons(filename):
    """~8 metadata rows — find first row where col 0 is a 4-digit year."""
    raw = pd.read_csv(RAW_DIR / filename, header=None, dtype=str)
    data_start = raw[raw[0].str.match(r'^\d{4}', na=False)].index[0]
    df = raw.iloc[data_start:].rename(columns={0: 'date', 1: 'value'}).copy()
    df['date']  = pd.to_datetime(df['date'], format='%Y %b', errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)


def load_fred(filename):
    """Clean 2-column format. Missing values stored as '.'."""
    df = pd.read_csv(RAW_DIR / filename, parse_dates=['DATE'])
    df.columns = ['date', 'value']
    df['value'] = pd.to_numeric(df['value'], errors='coerce')  # '.' -> NaN
    return df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)


def load_hpi(filename):
    """Large file — load UK national row only, normalise column names."""
    df = pd.read_csv(RAW_DIR / filename, low_memory=False)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    df = df[df['regionname'] == 'United Kingdom'].copy()
    df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
    return df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)


print('Loader functions ready')

Loader functions ready


In [2]:
series = {
    'base_rate':           load_boe('boe_base_rate.csv'),
    'mortgage_approvals':  load_boe('boe_mortgage_approvals.csv'),
    'consumer_credit':     load_boe('boe_consumer_credit.csv'),
    'sme_lending':         load_boe('boe_sme_lending.csv'),
    'cpi_inflation':       load_ons('ons_cpi_inflation.csv'),
    'gdp_growth':          load_ons('ons_gdp_growth.csv'),
    'unemployment_rate':   load_ons('ons_unemployment_rate.csv'),
    'avg_weekly_earnings': load_ons('ons_avg_weekly_earnings.csv'),
    'gilt_10yr':           load_fred('fred_gilt_10yr.csv'),
    'gbp_usd':             load_fred('fred_gbp_usd.csv'),
    'uk_cpi_oecd':         load_fred('fred_uk_cpi_oecd.csv'),
}

# HPI handled separately — extra columns kept
hpi = load_hpi('hpi_full_dataset.csv')

for name, df in series.items():
    print(f'  {name:25s}  {len(df):>4} rows   {df["date"].min().date()} -> {df["date"].max().date()}')

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/boe_base_rate.csv'

In [ ]:
def to_monthly(df, col_name, agg='mean'):
    """Resample a [date, value] DataFrame to month-end frequency."""
    df = df.set_index('date')[['value']].copy()
    df = df[~df.index.duplicated(keep='last')]  # keep latest revision per date
    df = df.resample('ME').agg(agg)
    df.columns = [col_name]
    return df

monthly = {}
for name, df in series.items():
    # GBP/USD is daily — monthly mean is correct aggregation
    # All others already monthly — resample just aligns day-of-month to month-end
    monthly[name] = to_monthly(df, name)

# GDP is quarterly — forward-fill to monthly
# Decision: ffill is correct because GDP represents the whole quarter
monthly['gdp_growth'] = monthly['gdp_growth'].resample('ME').ffill()

print('All series resampled to month-end')

In [ ]:
# Use 2010 as practical start — most series available, avoids wall of NaNs
# (SME lending starts 2011; pre-2010 rows would be mostly empty)
date_spine = pd.DataFrame(
    index=pd.date_range(
        start='2010-01-31',
        end=pd.Timestamp.today().normalize() + pd.offsets.MonthEnd(0),
        freq='ME'
    )
)

master = date_spine.copy()
for name, df in monthly.items():
    master = master.join(df, how='left')

# Join HPI columns separately
hpi_cols = hpi[['date', 'averageprice', 'index', 'salesvolume']].copy()
hpi_cols = hpi_cols.rename(columns={
    'averageprice': 'avg_price',
    'index':        'hpi_index',
    'salesvolume':  'sales_volume'
})
hpi_cols = hpi_cols.set_index('date')
hpi_cols.index = hpi_cols.index + pd.offsets.MonthEnd(0)  # align to month-end

# VOLUME LAG: sales_volume lags price by ~2 months — shift to correct alignment
# Decision: shift(-2) so volume sits alongside the price it actually corresponds to
hpi_cols['sales_volume'] = hpi_cols['sales_volume'].shift(-2)

master = master.join(hpi_cols, how='left')
master.index.name = 'date'
master = master.reset_index()

print(f'Master table: {master.shape[0]} rows x {master.shape[1]} columns')
print(f'Date range:   {master["date"].min().date()} -> {master["date"].max().date()}')
master.tail(6)

In [ ]:
# Log null counts BEFORE any imputation — this is the audit record
gap_log = pd.DataFrame({
    'series':     master.columns[1:],
    'null_count': master.iloc[:, 1:].isna().sum().values,
    'null_pct':   (master.iloc[:, 1:].isna().mean().values * 100).round(1)
}).sort_values('null_pct', ascending=False)

print('Null counts BEFORE imputation:')
display(gap_log)

gap_log.to_csv('../data/processed/gap_log_pre_imputation.csv', index=False)

In [ ]:
# Decision: interpolate gaps <= 3 months linearly
# Gaps > 3 months: leave as NaN and add a flag column for the quality report

INTERP_LIMIT = 3
value_cols = [c for c in master.columns if c != 'date']

for col in value_cols:
    # Flag long gaps BEFORE filling so the information is not lost
    null_mask   = master[col].isna()
    runs        = null_mask.ne(null_mask.shift()).cumsum()
    run_lengths = runs.map(runs[null_mask].value_counts())
    master[f'{col}_gap_flag'] = null_mask & (run_lengths > INTERP_LIMIT)

    # Interpolate short gaps only
    master[col] = master[col].interpolate(
        method='linear',
        limit=INTERP_LIMIT,
        limit_direction='forward'
    )

print(f'Interpolation done. Gap flags added for runs > {INTERP_LIMIT} months.')
print(f'Master table now: {master.shape[0]} rows x {master.shape[1]} columns')

In [ ]:
# Derived metrics
master['real_rate']          = master['base_rate'] - master['cpi_inflation']
master['affordability_ratio']= master['avg_price'] / (master['avg_weekly_earnings'] * 52)
master['credit_spread']      = master['gilt_10yr'] - master['base_rate']
master['hpi_yoy_pct']        = master['hpi_index'].pct_change(12) * 100

# Save
master.to_csv('../data/processed/master.csv', index=False)
print(f'Saved -> data/processed/master.csv  ({master.shape[0]} rows x {master.shape[1]} cols)')
master.tail(6)